# Польза посткалибровки вероятностей эмоций

Финальный тест — только актёры 17–20. Калибратор обучен на 21–24, модели — на 1–16.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

ROOT = Path('..')
metrics = pd.read_csv(ROOT / 'reports/calibration_metrics.csv')
confusion = pd.read_csv(ROOT / 'reports/confusion_matrices.csv')
metrics

In [ ]:
# Δ считается как after - before: для Log Loss/Brier/ECE отрицательное значение лучше.
display(metrics.style.format({c: '{:.4f}' for c in metrics.columns if c not in ['model', 'method']}))
after = metrics[metrics.method.ne('none')].copy()
best_model = after.sort_values('log_loss').iloc[0]['model']
print(f'Лучшая откалиброванная модель по Log Loss: {best_model}')

In [ ]:
# Функциональное сравнение качества вероятностей: меньше — лучше.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes, ['log_loss', 'brier_score', 'ece'], ['Log Loss', 'Multiclass Brier Score', 'ECE']):
    metrics.pivot_table(index='model', columns='method', values=metric).plot(kind='bar', ax=ax, color=['#8da0cb', '#fc8d62', '#66c2a5', '#e78ac3'])
    ax.set_title(title + ' (меньше — лучше)')
    ax.set_xlabel('')
    ax.set_ylabel('Значение')
    ax.tick_params(axis='x', rotation=25)
    ax.legend(['Без калибровки', 'Sigmoid', 'Isotonic', 'Temperature'])
plt.tight_layout()

In [ ]:
# Accuracy и Macro F1 — контроль того, изменилась ли граница решений.
fig, ax = plt.subplots(figsize=(9, 4))
quality = metrics.melt(id_vars=['model', 'method'], value_vars=['accuracy', 'macro_f1'], var_name='metric', value_name='value')
quality['label'] = quality.metric.map({'accuracy': 'Accuracy', 'macro_f1': 'Macro F1'})
for (model, label), part in quality.groupby(['model', 'label']):
    ax.plot(part.method, part.value, marker='o', label=f'{model} — {label}')
ax.set(title='Классовые метрики до и после калибровки', ylabel='Значение', xlabel='Этап')
ax.set_xticks(range(4), ['Без', 'Sigmoid', 'Isotonic', 'Temperature'])
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(alpha=.25)
plt.tight_layout()

In [ ]:
# Reliability diagram строится по сохранённым предсказаниям артефактов на тесте.
# Для воспроизводимости ниже показан компактный код повторной загрузки features и артефактов.
import joblib
from emotion_calibration.classify import feature_columns
features = pd.read_csv(ROOT / 'data/processed/features.csv')
test = features[features.actor.between(17, 20)]
cols = feature_columns(features)
model_key = best_model.lower().replace(' ', '_')
base = joblib.load(ROOT / 'reports/models' / f'{model_key}_none.joblib')
best_method = metrics[(metrics.model == best_model) & metrics.method.ne('none')].sort_values('log_loss').iloc[0]['method']
calibrated = joblib.load(ROOT / 'reports/models' / f'{model_key}_{best_method}.joblib')
y = test.emotion.to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, estimator, title in zip(axes, [base, calibrated], ['Без калибровки', 'Лучший метод']):
    proba = estimator.predict_proba(test[cols]) if title == 'Без калибровки' or best_method != 'temperature' else estimator.predict_proba(base.predict_proba(test[cols]))
    # Средняя one-vs-rest reliability curve по восьми эмоциям.
    curves = []
    for i, label in enumerate(estimator.classes_):
        truth = (y == label).astype(int)
        fraction, mean_prob = calibration_curve(truth, proba[:, i], n_bins=10, strategy='uniform')
        ax.plot(mean_prob, fraction, marker='o', alpha=.7, label=label)
    ax.plot([0, 1], [0, 1], '--', color='black')
    ax.set(title=f'{best_model}: {title}', xlabel='Средняя предсказанная вероятность', ylabel='Доля положительных')
    ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
# Распределение максимальной уверенности: видно, стала ли модель менее самоуверенной.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, estimator, title in zip(axes, [base, calibrated], ['До', 'После']):
    confidence = estimator.predict_proba(test[cols]).max(axis=1)
    ax.hist(confidence, bins=10, range=(0, 1), color='#66c2a5', edgecolor='white')
    ax.set(title=f'Уверенность: {title}', xlabel='Максимальная вероятность', ylabel='Количество записей')
    ax.grid(axis='y', alpha=.25)
plt.tight_layout()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, phase in zip(axes, ['none', best_method]):
    part = confusion[(confusion.model == best_model) & (confusion.phase == phase)]
    matrix = part.pivot(index='actual', columns='predicted', values='count').fillna(0)
    ConfusionMatrixDisplay(matrix.to_numpy(), display_labels=matrix.index).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
    ax.set_title(f'{best_model}: {phase}')
plt.tight_layout()

## Вывод

Считаем посткалибровку полезной, если после неё на независимых актёрах уменьшаются Log Loss, multiclass Brier Score и ECE. Accuracy и Macro F1 — контрольные метрики и могут не измениться. Интерпретация ограничена актёрской речью RAVDESS, 240 примерами калибровки и возможным дисбалансом интенсивности.